# 01 — Exploratory Data Analysis
## Loan Default Prediction System

**Purpose:** Perform an initial data audit of the raw loan application dataset — structure, missing values, class balance, distributions, and outliers — before any cleaning or modeling begins, per the project blueprint (Section 4, Machine Learning Module).

**Input:** `ml/data/raw/Loan_default.csv`
**Output:** Analysis only (no files written) — findings inform `02_feature_engineering.ipynb` and `ml/src/data_cleaning.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = ML_ROOT / "data" / "raw" / "Loan_default.csv"

print(f"Loading dataset from: {RAW_DATA_PATH}")
df = pd.read_csv(RAW_DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 1. Structural Overview

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Missing Value Audit

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(3)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report[missing_report["missing_count"] > 0] if missing_report["missing_count"].sum() > 0 else print("No missing values detected in the raw dataset.")

## 3. Duplicate Records

In [ ]:
n_full_duplicates = df.duplicated().sum()
n_id_duplicates = df["LoanID"].duplicated().sum() if "LoanID" in df.columns else None
print(f"Fully duplicated rows: {n_full_duplicates}")
print(f"Duplicate LoanID values: {n_id_duplicates}")

## 4. Target Variable — Class Balance

The target `Default` (1 = defaulted, 0 = did not default) is the label the entire pipeline is built around. Class imbalance here directly informs the modeling strategy (class weighting, recall-focused evaluation) specified in the blueprint.

In [ ]:
target_counts = df["Default"].value_counts()
target_pct = df["Default"].value_counts(normalize=True).round(4) * 100

print(target_counts)
print()
print(target_pct)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="Default", data=df, palette=["#2563eb", "#dc2626"], ax=ax)
ax.set_xticklabels(["No Default (0)", "Default (1)"])
ax.set_title("Target Class Distribution")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

## 5. Numeric Feature Distributions

In [ ]:
numeric_cols = ["Age", "Income", "LoanAmount", "CreditScore", "MonthsEmployed",
                "NumCreditLines", "InterestRate", "LoanTerm", "DTIRatio"]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#2563eb")
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 6. Numeric Features vs. Target

Comparing distributions by default status highlights which numeric features carry the strongest separating signal.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(x="Default", y=col, data=df, ax=ax, palette=["#2563eb", "#dc2626"])
    ax.set_xticklabels(["No Default", "Default"])
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 7. Outlier Detection (IQR Method)

Identifying the proportion of extreme values in key numeric fields — this quantifies what `data_cleaning.py`'s `handle_outliers` function will later cap.

In [ ]:
outlier_cols = ["Income", "LoanAmount", "InterestRate", "DTIRatio"]
outlier_summary = []

for col in outlier_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        "column": col, "lower_bound": round(lower, 2), "upper_bound": round(upper, 2),
        "n_outliers": int(n_outliers), "pct_outliers": round(n_outliers / len(df) * 100, 3)
    })

pd.DataFrame(outlier_summary)

## 8. Categorical Feature Distributions

In [ ]:
categorical_cols = ["Education", "EmploymentType", "MaritalStatus",
                    "HasMortgage", "HasDependents", "LoanPurpose", "HasCoSigner"]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for ax, col in zip(axes.flatten(), categorical_cols):
    sns.countplot(y=col, data=df, ax=ax, order=df[col].value_counts().index, color="#2563eb")
    ax.set_title(col)
axes.flatten()[-1].axis("off")
plt.tight_layout()
plt.show()

## 9. Categorical Features vs. Default Rate

This reveals which categories (e.g. employment type, loan purpose) are associated with materially higher default rates — directly relevant to feature engineering and model interpretability for loan officers.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for ax, col in zip(axes.flatten(), categorical_cols):
    default_rate = df.groupby(col)["Default"].mean().sort_values(ascending=False)
    sns.barplot(x=default_rate.values, y=default_rate.index, ax=ax, color="#dc2626")
    ax.set_title(f"Default Rate by {col}")
    ax.set_xlabel("Default Rate")
axes.flatten()[-1].axis("off")
plt.tight_layout()
plt.show()

## 10. Correlation Analysis (Numeric Features)

In [ ]:
corr_matrix = df[numeric_cols + ["Default"]].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix — Numeric Features & Target")
plt.tight_layout()
plt.show()

print("\nCorrelation with Default (sorted):")
print(corr_matrix["Default"].drop("Default").sort_values(key=abs, ascending=False))

## 11. EDA Summary & Findings

- **Dataset size:** 255,347 rows, 18 columns; no missing values and no duplicate records in this snapshot.
- **Class imbalance:** ~88.4% No Default vs. ~11.6% Default — this drives the use of `class_weight="balanced"` / `scale_pos_weight` and a recall-focused evaluation strategy in `train_pipeline.py`.
- **Outliers:** A small percentage of extreme values exist in `Income`, `LoanAmount`, `InterestRate`, and `DTIRatio` — these are capped (winsorized) rather than dropped in `data_cleaning.py`, preserving tail-risk records the model needs to learn from.
- **Signal features:** `CreditScore`, `InterestRate`, `DTIRatio`, and `MonthsEmployed` show the strongest association with `Default`, consistent with domain expectations.
- **Categorical risk drivers:** `EmploymentType` (particularly Unemployed) and `LoanPurpose` show visibly different default rates across categories, supporting one-hot encoding rather than dropping these fields.

**Next step:** `02_feature_engineering.ipynb` — derive `LoanToIncomeRatio`, `EmploymentStabilityRatio`, and `CreditScoreBand`, and validate the encoding/scaling strategy before training.